# PCP 套利实时监控（Jupyter 版）

**使用方法**：
1. 确认 Wind 终端已登录
2. 顺序运行所有 Cell（`Run All`）
3. 最后一个 Cell 进入轮询，刷新间隔 5 秒
4. 按 **■（Interrupt Kernel）** 停止监控

---
**参数配置见 Cell 2**

In [ ]:
# ── Cell 1：环境初始化 ──────────────────────────────────────────────
import sys, math, time
from datetime import datetime, date, timedelta
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

import pandas as pd
from IPython.display import display, clear_output, HTML

PROJECT_ROOT = Path(r'd:\Option_Arbitrage_Engine')
sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import get_default_config
from data_engine.contract_info import ContractInfoManager
from models import (
    ContractInfo, ETFTickData, TickData,
    OptionType, SignalType, TradeSignal, normalize_code,
)
from strategies.pcp_arbitrage import PCPArbitrage

print('✅ 依赖导入完成')

In [ ]:
# ── Cell 2：参数配置（根据需要修改）─────────────────────────────────

REFRESH_SECS     = 5      # 刷新间隔（秒）
MIN_DISPLAY_PROFIT = 30   # 显示阈值（元/组）；调高可减少噪音
MAX_EXPIRY_DAYS  = 90     # 只看N天内到期合约（90=近3个月）
ATM_RANGE_PCT    = 0.20   # ATM距离过滤：只看 ±20% 行权价
WIND_BATCH_SIZE  = 300    # Wind wsq 单批代码数（实测194个单次调用无问题）

CONTRACT_INFO_CSV = PROJECT_ROOT / 'info_data' / '上交所期权基本信息.csv'
WIND_OPT_FIELDS   = 'rt_last,rt_ask1,rt_bid1'  # 3字段（194x3=582点），rt_oi/ask_vol需L2权限
WIND_ETF_FIELDS   = 'rt_last,rt_ask1,rt_bid1'
ETF_NAME_MAP = {
    '510050': '50ETF', '510300': '300ETF', '510500': '500ETF',
    '588000': '科创50', '588050': '科创板50',
}

print(f'配置：刷新={REFRESH_SECS}s | 最小利润={MIN_DISPLAY_PROFIT}元 | 到期≤{MAX_EXPIRY_DAYS}天')

In [ ]:
# ── Cell 3：连接 Wind ─────────────────────────────────────────────
from WindPy import w
result = w.start()
if result.ErrorCode == 0:
    print('✅ Wind 连接成功')
else:
    print(f'❌ Wind 连接失败，错误码: {result.ErrorCode}')
    print('请先打开并登录 Wind 金融终端')

In [ ]:
# ── Cell 4：工具函数 ──────────────────────────────────────────────

def _fval(d, key, default=math.nan):
    v = d.get(key)
    if v is None: return default
    try:
        f = float(v)
        return default if math.isnan(f) else f
    except: return default

def _ival(d, key, default=0):
    v = d.get(key)
    if v is None: return default
    try: return int(float(v))
    except: return default

def poll_snapshot(codes, fields, cancel_first=False):
    """
    批量拉取 Wind 行情快照，返回 {code: {FIELD: value}}
    cancel_first=True：先取消旧订阅（每轮轮询第一次调用时使用，防止 -40522007）
    """
    if cancel_first:
        try: w.cancelRequest(0)
        except: pass
    out = {}
    for i in range(0, len(codes), WIND_BATCH_SIZE):
        if i > 0:
            try: w.cancelRequest(0)  # 每批前取消旧订阅，防止累积超限
            except: pass
        batch = codes[i:i+WIND_BATCH_SIZE]
        result = w.wsq(','.join(batch), fields)
        if result is None or result.ErrorCode != 0:
            continue
        field_names = [f.upper() for f in result.Fields]
        for j, raw_code in enumerate(result.Codes):
            row = {}
            for k, fn in enumerate(field_names):
                try: row[fn] = result.Data[k][j]
                except: row[fn] = None
            out[normalize_code(raw_code, '.SH')] = row
    return out

def make_option_tick(code, q, ts):
    last = _fval(q, 'RT_LAST', 0.0)
    ask1 = _fval(q, 'RT_ASK1')
    bid1 = _fval(q, 'RT_BID1')
    if last <= 0 or math.isnan(ask1) or math.isnan(bid1) or ask1 <= 0 or bid1 <= 0:
        return None
    return TickData(
        timestamp=ts, contract_code=code,
        current=last, volume=0, high=last, low=last, money=0.0,
        position=_ival(q, 'RT_OI'),
        ask_prices=[ask1]+[math.nan]*4,
        ask_volumes=[100]+[0]*4,  # rt_ask_vol1 需L2权限，默认100不影响计算
        bid_prices=[bid1]+[math.nan]*4,
        bid_volumes=[100]+[0]*4,
    )

def make_etf_tick(code, q, ts):
    last = _fval(q, 'RT_LAST', 0.0)
    if last <= 0: return None
    return ETFTickData(
        timestamp=ts, etf_code=code, price=last,
        ask_price=_fval(q, 'RT_ASK1'),
        bid_price=_fval(q, 'RT_BID1'),
        is_simulated=False,
    )

print('✅ 工具函数定义完成')

In [ ]:
# ── Cell 5：加载合约 + 构建配对 ──────────────────────────────────

config = get_default_config()
config.min_profit_threshold = MIN_DISPLAY_PROFIT
strategy = PCPArbitrage(config)

contract_mgr = ContractInfoManager()
n_contracts = contract_mgr.load_from_csv(CONTRACT_INFO_CSV)
print(f'已加载 {n_contracts} 条合约信息')

today = date.today()
active = [
    info for info in contract_mgr.contracts.values()
    if info.list_date <= today <= info.expiry_date
    and (info.expiry_date - today).days <= MAX_EXPIRY_DAYS
]
print(f'活跃合约（{MAX_EXPIRY_DAYS}天内到期）: {len(active)} 个')

# ETF 代码列表
etf_codes = sorted(set(c.underlying_code for c in active))
print(f'标的 ETF: {etf_codes}')

# 初始 ETF 价格（用于 ATM 过滤）
print('\n拉取 ETF 初始价格...')
etf_snap = poll_snapshot(etf_codes, WIND_ETF_FIELDS)
etf_prices = {}
for code in etf_codes:
    q = etf_snap.get(code, {})
    px = _fval(q, 'RT_LAST', 0.0)
    name = ETF_NAME_MAP.get(code.split('.')[0], code)
    if px > 0:
        etf_prices[code] = px
        print(f'  {name}: {px:.4f}')
    else:
        strikes = [c.strike_price for c in active if c.underlying_code == code]
        if strikes:
            etf_prices[code] = (min(strikes)+max(strikes))/2
            print(f'  {name}: 估算 {etf_prices[code]:.4f}（非交易时间或无行情）')

# 构建 Call/Put 配对
pairs = []
option_codes = set()
by_und = defaultdict(lambda: defaultdict(list))
for info in active:
    by_und[info.underlying_code][info.expiry_date].append(info)

for u_code, expiry_map in by_und.items():
    etf_px = etf_prices.get(u_code, 0.0)
    for expiry, contracts in expiry_map.items():
        calls = {c.strike_price: c for c in contracts if c.option_type == OptionType.CALL}
        puts  = {c.strike_price: c for c in contracts if c.option_type == OptionType.PUT}
        for strike in sorted(set(calls) & set(puts)):
            if etf_px > 0 and abs(strike-etf_px)/etf_px > ATM_RANGE_PCT:
                continue
            pairs.append((calls[strike], puts[strike]))
            option_codes.add(calls[strike].contract_code)
            option_codes.add(puts[strike].contract_code)

option_codes = list(option_codes)
print(f'\nCall/Put 配对: {len(pairs)} 组  订阅期权: {len(option_codes)} 个')
print('\n准备就绪，运行下一个 Cell 开始监控 ▶')

In [ ]:
# ── Cell 6：主监控循环（按 ■ 中断内核停止）──────────────────────────
#
# 显示说明：
#   正向套利（Conversion）= 买ETF + 买Put + 卖Call，持有至到期
#   反向套利（Reversal）  = 卖ETF + 卖Put + 买Call（A股受T+1限制）
#   净利润 = 扣除手续费+滑点后的理论无风险收益（每组合约，单位：元）

def _color_profit(val):
    """净利润数值颜色映射"""
    if val >= 200: return 'background-color: #00aa44; color: white; font-weight: bold'
    if val >= 100: return 'background-color: #44bb66; color: white'
    if val >= 50:  return 'background-color: #88dd88; color: black'
    return 'background-color: #ccffcc; color: black'

def _color_deviation(val):
    if val > 0:  return 'color: green'
    if val < 0:  return 'color: red'
    return ''

iteration = 0
etf_display = dict(etf_prices)

try:
    while True:
        ts = datetime.now()

        # -- 拉取 ETF 行情（先取消旧订阅，防止 -40522007 累积超限）--
        etf_snap = poll_snapshot(etf_codes, WIND_ETF_FIELDS, cancel_first=True)
        for code, q in etf_snap.items():
            tick = make_etf_tick(code, q, ts)
            if tick:
                strategy.on_etf_tick(tick)
                etf_display[code] = tick.price

        # -- 拉取期权行情（分批）--
        opt_snap = poll_snapshot(option_codes, WIND_OPT_FIELDS)
        for code, q in opt_snap.items():
            tick = make_option_tick(code, q, ts)
            if tick:
                strategy.on_option_tick(tick)

        # -- 扫描套利信号 --
        signals = strategy.scan_opportunities(pairs, current_time=ts)
        signals = [s for s in signals if s.net_profit_estimate >= MIN_DISPLAY_PROFIT]
        iteration += 1

        # -- 刷新显示 --
        clear_output(wait=True)

        # 标题行
        etf_str = '  |  '.join(
            f"<b style='color:#0077cc'>{ETF_NAME_MAP.get(c.split('.')[0],c)}</b>"
            f" <span style='color:#ff8800;font-size:1.1em'>{p:.4f}</span>"
            for c, p in etf_display.items() if p > 0
        )
        header_html = f"""
        <div style='font-family:monospace; padding:8px; background:#1a1a2e; color:white; border-radius:6px; margin-bottom:8px'>
          <span style='color:#00ff88; font-size:1.2em; font-weight:bold'>⚡ PCP 套利实时监控</span>
          &nbsp;&nbsp;
          <span style='color:#aaa; font-size:0.9em'>{ts.strftime('%H:%M:%S')}</span>
          &nbsp; 第 {iteration} 次刷新
          <br>
          {etf_str}
          <br>
          <span style='color:#888; font-size:0.85em'>
            监控配对: {len(pairs)} 组 &nbsp;|
            订阅期权: {len(option_codes)} 个 &nbsp;|
            套利信号 (≥{MIN_DISPLAY_PROFIT}元): <b style='color:#ffff00'>{len(signals)}</b> 条
          </span>
        </div>
        """
        display(HTML(header_html))

        if signals:
            rows = []
            for sig in signals[:30]:
                direction = '正向' if sig.signal_type == SignalType.FORWARD else '反向'
                pcp_dev = sig.actual_spread - sig.theoretical_spread
                u_name = ETF_NAME_MAP.get(sig.underlying_code.split('.')[0], sig.underlying_code)
                rows.append({
                    '到期日':   sig.expiry.strftime('%m-%d'),
                    '标的':     u_name,
                    '行权价':   round(sig.strike, 4),
                    '方向':     direction,
                    'Call买1':  round(sig.call_bid, 4),
                    'Call卖1':  round(sig.call_ask, 4),
                    'Put买1':   round(sig.put_bid, 4),
                    'Put卖1':   round(sig.put_ask, 4),
                    'ETF现价':  round(sig.spot_price, 4),
                    'PCP偏差':  round(pcp_dev, 4),
                    '净利润(元)': round(sig.net_profit_estimate, 0),
                    '置信度':   round(sig.confidence, 2),
                    'Call代码': sig.call_code,
                    'Put代码':  sig.put_code,
                })

            df = pd.DataFrame(rows)

            # 应用样式
            styled = (
                df.style
                .applymap(_color_profit,      subset=['净利润(元)'])
                .applymap(_color_deviation,   subset=['PCP偏差'])
                .set_properties(**{'font-family': 'monospace', 'font-size': '0.9em'})
                .set_table_styles([{
                    'selector': 'thead tr th',
                    'props': [('background', '#2d3561'), ('color', 'white'),
                              ('font-weight', 'bold'), ('padding', '6px 10px')]
                }])
                .hide(axis='index')
            )
            display(styled)

            # 最优机会操作指引
            top = signals[0]
            direction = '正向' if top.signal_type == SignalType.FORWARD else '反向'
            u_name = ETF_NAME_MAP.get(top.underlying_code.split('.')[0], top.underlying_code)
            if top.signal_type == SignalType.FORWARD:
                guide = (
                    f"<b style='color:#00ff88'>【最优机会：正向套利（Conversion）】</b>"
                    f" 预估净利润 <b style='color:#ffaa00'>{top.net_profit_estimate:.0f} 元/组</b>"
                    f" | 到期: {top.expiry} | 行权价: {top.strike:.4f}<br>"
                    f"&nbsp;&nbsp;① 卖出 <b>{u_name} {top.strike:.4f} 认购</b>"
                    f" ({top.call_code})&nbsp; 参考价 ≥ <b style='color:#ffff00'>{top.call_bid:.4f}</b><br>"
                    f"&nbsp;&nbsp;② 买入 <b>{u_name} {top.strike:.4f} 认沽</b>"
                    f" ({top.put_code})&nbsp; 参考价 ≤ <b style='color:#ffff00'>{top.put_ask:.4f}</b><br>"
                    f"&nbsp;&nbsp;③ 买入 <b>{top.underlying_code.replace('.SH','')} ETF</b>"
                    f"&nbsp; 参考价 ≤ <b style='color:#ffff00'>{top.spot_price:.4f}</b>"
                )
            else:
                guide = (
                    f"<b style='color:#00aaff'>【最优机会：反向套利（Reversal）】</b>"
                    f" 预估净利润 <b style='color:#ffaa00'>{top.net_profit_estimate:.0f} 元/组</b>"
                    f" | 到期: {top.expiry} | 行权价: {top.strike:.4f}<br>"
                    f"&nbsp;&nbsp;① 买入 <b>{u_name} {top.strike:.4f} 认购</b>"
                    f" ({top.call_code})&nbsp; 参考价 ≤ <b style='color:#ffff00'>{top.call_ask:.4f}</b><br>"
                    f"&nbsp;&nbsp;② 卖出 <b>{u_name} {top.strike:.4f} 认沽</b>"
                    f" ({top.put_code})&nbsp; 参考价 ≥ <b style='color:#ffff00'>{top.put_bid:.4f}</b><br>"
                    f"&nbsp;&nbsp;③ 卖空 <b>{top.underlying_code.replace('.SH','')} ETF</b>"
                    f"&nbsp; <span style='color:#aaa'>（A股 ETF 受 T+1 限制，谨慎评估可行性）</span>"
                )

            display(HTML(
                f"<div style='border:1px solid #555; border-radius:4px; padding:10px; "
                f"margin-top:8px; font-family:monospace; background:#0d1117; color:#ddd'>"
                f"{guide}</div>"
            ))

        else:
            display(HTML(
                f"<div style='color:#888; font-family:monospace; padding:12px'>"
                f"暂无套利机会（净利润阈值 ≥ {MIN_DISPLAY_PROFIT} 元/组）</div>"
            ))

        time.sleep(REFRESH_SECS)

except KeyboardInterrupt:
    clear_output(wait=True)
    print('✅ 监控已停止')
    if 'signals' in dir() and signals:
        print(f'最后一次扫描：{len(signals)} 条套利信号')